<img src="images/14.png" width="40%">

<img src="images/15.png" width="40%">

<img src="images/16.png" width="40%">

一个完整的softmax实现：
加载数据集，图片转为张量
初始化回归参数，d = 784  # 特征维度   K = 10   # 类别数量
交叉熵损失函数，接收logits，内部自带softmax+ -log(p)（内置torch.nn.CrossEntropyLoss()）
训练循环：
线性计算 Z = XW + b
损失：loss_i = -log(ŷ)，CrossEntropy内部完成softmax
反向传播求梯度
参数更新
梯度清零
每轮epoch输出训练loss和准确率accuracy

In [5]:
import torch
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor()
])

mnist_train = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
mnist_test = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

batch_size = 18
train_loader = torch.utils.data.DataLoader(mnist_train, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(mnist_test, batch_size=batch_size, shuffle=False)

d = 784
K = 10

W = torch.normal(mean=0.0, std=0.01, size=(d, K), requires_grad=True)
b = torch.zeros(K, requires_grad=True)

learning_rate = 0.01
epochs = 5
loss_fn = torch.nn.CrossEntropyLoss()

for epoch in range(epochs):
    total_loss = 0.0
    correct = 0
    total_samples = 0

    for X_batch, y_batch in train_loader:
        X_flat = X_batch.reshape(-1, d)
        Z = torch.matmul(X_flat, W) + b
        loss = loss_fn(Z, y_batch)

        loss.backward()

        # 参数更新：no_grad内原地修改叶子，合法
        with torch.no_grad():
            W -= learning_rate * W.grad
            b -= learning_rate * b.grad

        W.grad.zero_()
        b.grad.zero_()

        total_loss += loss.item()
        pred = torch.argmax(Z, dim=1)
        correct += (pred == y_batch).sum().item()
        total_samples += y_batch.shape[0]

    train_acc = correct / total_samples
    print(f"Epoch {epoch+1:2d} | Loss: {total_loss:.4f} | Train Acc: {train_acc:.4f}")

# 测试评估
with torch.no_grad():
    correct_test = 0
    total_test = 0
    for X_batch, y_batch in test_loader:
        X_flat = X_batch.reshape(-1, d)
        Z = torch.matmul(X_flat, W) + b
        pred = torch.argmax(Z, dim=1)
        correct_test += (pred == y_batch).sum().item()
        total_test += y_batch.shape[0]
    test_acc = correct_test / total_test
    print(f"\n====测试集准确率 Test Accuracy = {test_acc:.4f} ====")


Epoch  1 | Loss: 2115.0193 | Train Acc: 0.8560
Epoch  2 | Loss: 1328.4350 | Train Acc: 0.8931
Epoch  3 | Loss: 1203.0412 | Train Acc: 0.9002
Epoch  4 | Loss: 1138.5023 | Train Acc: 0.9049
Epoch  5 | Loss: 1096.8052 | Train Acc: 0.9081

====测试集准确率 Test Accuracy = 0.9156 ====


<img src="images/17.png" width="40%">

In [6]:
import torch
from torch import nn
from d2l import torch as d2l

# 1. 数据加载
batch_size = 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)

# 2. 构建网络
# nn.Sequential：顺序容器，数据依次流过里面的层
# nn.Flatten：展平，把 [batch, 1, 28, 28] 的图片变成 [batch,784]
# nn.Linear(784,10)：线性层，等价 Z = XW + b，输出10个logits
net = nn.Sequential(nn.Flatten(), nn.Linear(784,10))

# 3. 权重初始化函数
def init_weights(m):
    # 判断模块是不是Linear全连接层
    if type(m) == nn.Linear:
        # 权重：正态分布初始化，均值0，标准差0.01
        nn.init.normal_(m.weight, std=0.01)

# 遍历网络所有层，执行初始化
net.apply(init_weights)

# 4. 损失函数
# CrossEntropyLoss = softmax + 负对数交叉熵
# 直接输入logits，内部自动softmax，数值更稳定，**不要手动加softmax层**
loss = nn.CrossEntropyLoss()

# 5. 优化器 SGD
# net.parameters() 自动取出W和b，不用手动维护W,b
trainer = torch.optim.SGD(net.parameters(),lr=0.1)

# 6. 训练轮数
num_epochs = 10
# d2l封装好的训练函数：循环训练、计算loss、训练/测试集准确率、画图
d2l.train_ch3(net,train_iter,test_iter,loss,num_epochs,trainer)


ModuleNotFoundError: No module named 'd2l'